In [2]:
import pandas as pd
import psycopg
from dotenv import load_dotenv, find_dotenv

# Read the database settings from the .env file in the project folder
load_dotenv(find_dotenv(usecwd=True))

# Load one row per customer from the mart view
with psycopg.connect() as conn:
    cursor = conn.execute("SELECT * FROM marts.customer_outcomes")
    df = pd.DataFrame(cursor.fetchall(), columns=[c.name for c in cursor.description])

# Postgres NUMERIC values arrive as text-like objects; convert money columns to numbers
df = df.astype({"spend": float, "history": float})

df.shape

(64000, 18)

In [3]:
print(df.head(10))

   customer_id      arm  is_treated        segment  recency  recency_band  \
0            1   womens           1  Womens E-Mail       10  10-12 months   
1            2  control           0      No E-Mail        6  04-06 months   
2            3   womens           1  Womens E-Mail        7  07-09 months   
3            4     mens           1    Mens E-Mail        9  07-09 months   
4            5   womens           1  Womens E-Mail        2  01-03 months   
5            6   womens           1  Womens E-Mail        6  04-06 months   
6            7   womens           1  Womens E-Mail        9  07-09 months   
7            8   womens           1  Womens E-Mail        9  07-09 months   
8            9     mens           1    Mens E-Mail        9  07-09 months   
9           10   womens           1  Womens E-Mail       10  10-12 months   

   history history_segment  history_segment_order  mens  womens  \
0   142.44  2) $100 - $200                      2     1       0   
1   329.08  3) $20

In [4]:
check = df.groupby("arm").agg(
    customers=("customer_id", "count"),
    buyers=("conversion", "sum"),
)
check

,customers,buyers
arm,,
control,21306,122
mens,21307,267
womens,21387,189


In [5]:
from scipy.stats import chisquare

# Customers per arm; the test compares them with a perfect 1/3 split
customers = df["arm"].value_counts().sort_index()
stat, p_value = chisquare(customers)

print(customers.to_dict())
print("p-value:", round(p_value, 4))

{'control': 21306, 'mens': 21307, 'womens': 21387}
p-value: 0.9037


In [6]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests

def compare_rate(df, metric, arm, control="control"):
    """Compare one 0/1 metric (like conversion) between an email arm and control."""
    a = df.loc[df["arm"] == arm, metric]      # email group
    c = df.loc[df["arm"] == control, metric]  # control group

    # z-test: is the gap luck?
    z, p_value = proportions_ztest([a.sum(), c.sum()], [len(a), len(c)])

    # 95% confidence interval for the gap
    low, high = confint_proportions_2indep(
        a.sum(), len(a), c.sum(), len(c), compare="diff", method="wald"
    )

    return {
        "metric": metric,
        "arm": arm,
        "arm_rate_pct": a.mean() * 100,
        "control_rate_pct": c.mean() * 100,
        "lift_pp": (a.mean() - c.mean()) * 100,
        "ci_low_pp": low * 100,
        "ci_high_pp": high * 100,
        "p_value": p_value,
    }

results = pd.DataFrame([compare_rate(df, "conversion", arm) for arm in ["mens", "womens"]])

# Holm correction for testing two emails
results["p_value_holm"] = multipletests(results["p_value"], method="holm")[1]
results["not_luck"] = results["p_value_holm"] < 0.05

results.round(4)

,metric,arm,arm_rate_pct,control_rate_pct,lift_pp,ci_low_pp,ci_high_pp,p_value,p_value_holm,not_luck
0,conversion,mens,1.2531,0.5726,0.6805,0.5000,0.8610,0.0000,0.0000,True
1,conversion,womens,0.8837,0.5726,0.3111,0.1499,0.4723,0.0002,0.0002,True


In [7]:
visit_results = pd.DataFrame([compare_rate(df, "visit", arm) for arm in ["mens", "womens"]])
visit_results["p_value_holm"] = multipletests(visit_results["p_value"], method="holm")[1]
visit_results["not_luck"] = visit_results["p_value_holm"] < 0.05

visit_results.round(4)

,metric,arm,arm_rate_pct,control_rate_pct,lift_pp,ci_low_pp,ci_high_pp,p_value,p_value_holm,not_luck
0,visit,mens,18.2757,10.6167,7.6590,6.9954,8.3226,0.0,0.0,True
1,visit,womens,15.1400,10.6167,4.5233,3.8894,5.1572,0.0,0.0,True


In [8]:
import numpy as np
from scipy.stats import ttest_ind

def bootstrap_mean_diff(a, c, n_boot=10_000, seed=42):
    """Bootstrap 95% range for the difference in average spend (arm minus control)."""
    rng = np.random.default_rng(seed)   # seed = same random draws every run
    a = np.asarray(a)
    c = np.asarray(c)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        # Draw a new group of the same size, with replacement, from each arm
        a_sample = rng.choice(a, size=len(a), replace=True)
        c_sample = rng.choice(c, size=len(c), replace=True)
        diffs[i] = a_sample.mean() - c_sample.mean()
    # The middle 95% of the 10,000 differences is the range
    low, high = np.percentile(diffs, [2.5, 97.5])
    return a.mean() - c.mean(), low, high

control_spend = df.loc[df["arm"] == "control", "spend"]
rows = []
for arm in ["mens", "womens"]:
    arm_spend = df.loc[df["arm"] == arm, "spend"]

    lift, boot_low, boot_high = bootstrap_mean_diff(arm_spend, control_spend)

    # t-test as a check: Welch version (does not assume equal spread in both groups)
    t_result = ttest_ind(arm_spend, control_spend, equal_var=False)
    t_ci = t_result.confidence_interval()

    rows.append({
        "arm": arm,
        "lift_usd": lift,
        "bootstrap_low": boot_low,
        "bootstrap_high": boot_high,
        "ttest_low": t_ci.low,
        "ttest_high": t_ci.high,
        "ttest_p_value": t_result.pvalue,
    })

spend_results = pd.DataFrame(rows)
spend_results.round(4)

,arm,lift_usd,bootstrap_low,bootstrap_high,ttest_low,ttest_high,ttest_p_value
0,mens,0.7698,0.4864,1.0583,0.4851,1.0545,0.0000
1,womens,0.4244,0.1621,0.6841,0.1690,0.6799,0.0011
